# Structured Email Extraction — Zero Extra LLM Calls

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/structured_extraction.ipynb)

Commune can automatically extract structured JSON from inbound emails using a JSON Schema you define per inbox. No extra LLM call — the extraction happens before the webhook fires.

**Use cases:**
- Support ticket triage (extract: issue_type, priority, affected_product)
- Order confirmations (extract: order_id, items, total, shipping_address)
- Invoice processing (extract: invoice_number, vendor, amount, due_date)
- Job applications (extract: candidate_name, years_experience, skills)

**How it works:**
1. You define a JSON schema on an inbox
2. Commune parses every inbound email against the schema
3. Your webhook payload includes an `extracted_data` field with the parsed values
4. Your agent uses the structured data without any extra LLM parsing step

In [ ]:
!pip install commune-mail -q
print("\u2713 commune-mail installed")

## Setup

Initialize the Commune client and create a support inbox. We'll attach an extraction schema to this inbox so every inbound email is automatically parsed.

In [ ]:
import os
import json
from commune import CommuneClient

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")

commune = CommuneClient(api_key=COMMUNE_API_KEY)

# Create the inbox that will receive support tickets
support_inbox = commune.inboxes.create(local_part="structured-support")

print(f"\u2713 Inbox created: {support_inbox.address}")
print(f"  Inbox ID: {support_inbox.id}")

## Define the Extraction Schema

The schema is a standard JSON Schema object. Commune extracts values for every property you define — even if the email doesn't explicitly state them (the LLM infers from context).

Tips for writing good schemas:
- Use `enum` to constrain categorical values (issue_type, priority, etc.)
- Add `description` to each field — the extraction LLM reads these as instructions
- Keep the schema focused: 5-8 fields extracts faster and more accurately than 20+

In [ ]:
# Support ticket extraction schema
support_ticket_schema = {
    "type": "object",
    "properties": {
        "issue_type": {
            "type": "string",
            "enum": ["billing", "technical", "account", "shipping", "feature_request", "general"],
            "description": "The primary category of the customer's issue."
        },
        "priority": {
            "type": "string",
            "enum": ["low", "medium", "high", "critical"],
            "description": (
                "Urgency level. 'critical' = production down or financial loss. "
                "'high' = customer blocked. 'medium' = important but not blocking. "
                "'low' = general questions."
            )
        },
        "affected_product": {
            "type": "string",
            "description": "The specific product, feature, or service mentioned in the email."
        },
        "customer_sentiment": {
            "type": "string",
            "enum": ["positive", "neutral", "frustrated", "angry"],
            "description": "The overall emotional tone of the customer's message."
        },
        "order_id": {
            "type": "string",
            "description": "Order or transaction ID mentioned in the email, if any. Null if not present."
        },
        "requires_refund": {
            "type": "boolean",
            "description": "True if the customer is explicitly requesting a refund or chargeback."
        },
        "one_line_summary": {
            "type": "string",
            "description": "A single sentence summarizing the customer's core request."
        }
    },
    "required": ["issue_type", "priority", "customer_sentiment", "requires_refund", "one_line_summary"]
}

print("Extraction schema defined:")
print(json.dumps(support_ticket_schema, indent=2))

## Attach the Schema to the Inbox

Once the schema is set on an inbox, every subsequent inbound email will be automatically parsed. The `extracted_data` field appears in the webhook payload and in `message.extracted` when you fetch the message via the API.

In [ ]:
# Attach the schema to the inbox
updated_inbox = commune.inboxes.update(
    support_inbox.id,
    extraction_schema=support_ticket_schema,
)

print(f"\u2713 Extraction schema attached to {updated_inbox.address}")
print(f"  Schema fields: {list(support_ticket_schema['properties'].keys())}")
print()
print("Every email that arrives at this inbox will now be automatically")
print("parsed against this schema before your webhook fires.")

## What the Webhook Payload Looks Like

When Commune fires a webhook for an inbound email at this inbox, the payload includes the `extracted_data` field alongside the standard email fields. Your agent can use the structured data immediately — no extra LLM parsing step.

In [ ]:
# Example webhook payload for a billing complaint
# (This is what your webhook endpoint receives)
example_webhook_payload = {
    "event": "message.received",
    "inbox_id": support_inbox.id,
    "thread_id": "thread_example_001",
    "message_id": "msg_example_001",
    "sender": "customer@example.com",
    "subject": "Charged twice for order #4521 — please refund",
    "content": (
        "Hi, I was charged $49.99 twice for order #4521 on March 5th. "
        "My bank statement shows two identical charges from your company. "
        "I need the duplicate charge refunded immediately. This is extremely "
        "frustrating — I've been a customer for 3 years and this has never happened. "
        "Please resolve this today."
    ),
    # Commune automatically adds this field based on the inbox schema:
    "extracted_data": {
        "issue_type": "billing",
        "priority": "high",
        "affected_product": None,
        "customer_sentiment": "angry",
        "order_id": "#4521",
        "requires_refund": True,
        "one_line_summary": "Customer was charged twice for order #4521 and requests an immediate refund."
    }
}

print("Example webhook payload with extracted_data:")
print(json.dumps(example_webhook_payload, indent=2))

## Use Extracted Data to Route Tickets

With structured data, your agent can make routing decisions programmatically — no LLM call required. The extracted fields drive the logic directly.

In [ ]:
def route_support_ticket(webhook_payload: dict) -> dict:
    """Route an inbound support ticket based on extracted data.

    Uses the structured extracted_data field — no LLM call needed for routing.
    Returns a routing decision dict.
    """
    extracted = webhook_payload.get("extracted_data", {})

    issue_type = extracted.get("issue_type", "general")
    priority = extracted.get("priority", "low")
    requires_refund = extracted.get("requires_refund", False)
    sentiment = extracted.get("customer_sentiment", "neutral")
    summary = extracted.get("one_line_summary", "")

    # Routing logic: purely deterministic, no LLM
    routing = {
        "thread_id": webhook_payload["thread_id"],
        "sender": webhook_payload["sender"],
        "summary": summary,
        "team": None,
        "escalate_immediately": False,
        "auto_acknowledge": True,
        "tags": [],
    }

    # Assign to team based on issue type
    team_map = {
        "billing": "billing-team@company.com",
        "technical": "engineering@company.com",
        "account": "accounts@company.com",
        "shipping": "logistics@company.com",
        "feature_request": "product@company.com",
        "general": "support@company.com",
    }
    routing["team"] = team_map.get(issue_type, "support@company.com")

    # Escalate critical issues or refund requests from angry customers
    if priority == "critical" or (requires_refund and sentiment == "angry"):
        routing["escalate_immediately"] = True
        routing["tags"].append("escalated")

    # Tag based on issue type
    routing["tags"].append(issue_type)
    if requires_refund:
        routing["tags"].append("refund-requested")
    if sentiment in ("frustrated", "angry"):
        routing["tags"].append("unhappy-customer")

    return routing


# Run the router on our example payload
routing_decision = route_support_ticket(example_webhook_payload)

print("Routing decision (no LLM call needed):")
print(json.dumps(routing_decision, indent=2))

## Fetch Extracted Data from a Message

You can also retrieve the extracted data at any time by fetching the message object via the API. The `.extracted` field is always available after the message is processed.

In [ ]:
# Example: fetch a message and read its extracted fields
# (Replace 'msg_example_001' with a real message ID from your inbox)
fetch_example = '''
# Fetch a message by ID
message = commune.messages.get("msg_example_001")

# Access the extracted data
print(message.extracted)
# {
#   "issue_type": "billing",
#   "priority": "high",
#   "customer_sentiment": "angry",
#   "order_id": "#4521",
#   "requires_refund": True,
#   "one_line_summary": "Customer was charged twice..."
# }

# Use it directly in your agent logic
if message.extracted.get("requires_refund") and message.extracted.get("priority") == "high":
    notify_billing_team(message)
'''

print("Fetching extracted data from a message:")
print(fetch_example)

## Schema Examples for Other Use Cases

The same pattern works for any email type. Here are ready-to-use schemas for common scenarios.

In [ ]:
# Invoice processing schema
invoice_schema = {
    "type": "object",
    "properties": {
        "invoice_number": {
            "type": "string",
            "description": "Invoice number or ID from the email."
        },
        "vendor_name": {
            "type": "string",
            "description": "Name of the company or vendor sending the invoice."
        },
        "amount": {
            "type": "number",
            "description": "Total invoice amount as a number (e.g. 1250.00)."
        },
        "currency": {
            "type": "string",
            "description": "3-letter currency code (USD, EUR, GBP, etc.)."
        },
        "due_date": {
            "type": "string",
            "description": "Payment due date in ISO 8601 format (YYYY-MM-DD)."
        },
        "payment_terms": {
            "type": "string",
            "enum": ["immediate", "net_15", "net_30", "net_60", "net_90"],
            "description": "Payment terms from the invoice."
        }
    },
    "required": ["vendor_name", "amount", "currency"]
}

# Job application schema
job_application_schema = {
    "type": "object",
    "properties": {
        "candidate_name": {
            "type": "string",
            "description": "Full name of the job applicant."
        },
        "years_of_experience": {
            "type": "number",
            "description": "Total years of relevant professional experience."
        },
        "role_applied_for": {
            "type": "string",
            "description": "The specific role or position the candidate is applying for."
        },
        "skills": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of technical or professional skills mentioned."
        },
        "has_portfolio_link": {
            "type": "boolean",
            "description": "True if the email contains a link to a portfolio, GitHub, or work samples."
        },
        "availability": {
            "type": "string",
            "enum": ["immediate", "2_weeks", "1_month", "flexible", "unknown"],
            "description": "When the candidate is available to start."
        }
    },
    "required": ["candidate_name", "role_applied_for"]
}

print("Invoice schema fields:", list(invoice_schema["properties"].keys()))
print("Job application schema fields:", list(job_application_schema["properties"].keys()))
print()
print("To use either schema, call:")
print("  commune.inboxes.update(inbox_id, extraction_schema=<schema>)")

## What's next?

- **[Quickstart notebook](./quickstart.ipynb)** — basic inbox, send, and thread operations
- **[LangChain notebook](./langchain_customer_support.ipynb)** — use extracted data to drive a LangChain agent's routing logic
- **[CrewAI notebook](./crewai_multi_agent_crew.ipynb)** — route tickets to specialist agents based on `issue_type`
- **Semantic search** — after extraction, use `commune.search.threads()` to find similar past tickets automatically
- **Bulk processing** — apply schemas to existing inboxes to retroactively extract structured data from all historical threads

**Commune docs:** https://commune.email/docs  
**JSON Schema reference:** https://json-schema.org/understanding-json-schema